This one is about testing the DAE Only using normal FED AVG

In [70]:
### Imports ###

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
import torch.nn as nn
import logging
from torch.optim import AdamW
import copy
import joblib
import time
import json
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, accuracy_score, matthews_corrcoef, balanced_accuracy_score,
    roc_curve, precision_recall_curve, confusion_matrix
)

In [71]:
# ================= TON-IoT PREPROCESSING (ADD THIS CELL) =================

import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

DROP_COLUMNS = [
    "src_ip","dst_ip","dns_query","ssl_subject","ssl_issuer",
    "http_uri","http_user_agent","weird_name","weird_addl"
]

CATEGORICAL_COLUMNS = [
    "proto","service","conn_state","dns_qclass","dns_qtype",
    "dns_rcode","http_method","http_version","type"
]

def preprocess_toniot_dataframe(df, encoder=None, fit_encoder=False, feature_columns=None):

    df = df.copy()

    DROP_COLUMNS = [
        "src_ip","dst_ip","dns_query","ssl_subject","ssl_issuer",
        "http_uri","http_user_agent","weird_name","weird_addl"
    ]
    df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])

    df = df.replace("-", "missing")

    CATEGORICAL_COLUMNS = [
        "proto","service","conn_state","dns_qclass","dns_qtype",
        "dns_rcode","http_method","http_version","type"
    ]

    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            df[col] = df[col].astype(str)

    if encoder is None:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    if fit_encoder:
        df[CATEGORICAL_COLUMNS] = encoder.fit_transform(df[CATEGORICAL_COLUMNS])
    else:
        df[CATEGORICAL_COLUMNS] = encoder.transform(df[CATEGORICAL_COLUMNS])

    df = df.apply(pd.to_numeric, errors="coerce").fillna(0)

    # 🔒 Freeze schema
    if fit_encoder:
        feature_columns = df.columns.tolist()
    else:
        df = df.reindex(columns=feature_columns, fill_value=0)

    return df.values.astype(np.float32), encoder, feature_columns

In [72]:
## Configurations & Hyperparameters ###
NUM_CLIENTS = 20
BATCH_SIZE = 64
PREVIOUS_OPTIMAL_THRESHOLD = 0.0033788118
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 50
CLIENT_EPOCHS_PER_ROUND = 1
BATCH_SIZE = 1024
LR = 1e-3
WEIGHT_DECAY = 1e-4
LATENT_DIM = 16
DROPOUT_P = 0.1
L1_LAMBDA = 1e-5
INITIAL_NOISE = 0.05
FINAL_NOISE = 0.02
PATIENCE = 10
INPUT_DIM = 35
ENCODER_PATH = r"/home/azwad/Works/IoMT_FL/Previous_Implementation/Dataset/after_scaling_encoding"
DATA_PATH = "/home/azwad/Works/IoMT_FL/Revised_Implementation/data/CIC_IoMT_2024"
OUTPUT_DIR = "/home/azwad/Works/IoMT_FL/Revised_Implementation/results/FL_Centralized_FedAvg_Phase3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def setup_logging(log_file='training.log'):
    """Sets up a logger that writes to a file."""
    # Remove any existing handlers
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    
    # Create file handler
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setLevel(logging.INFO)
    
    # Create formatter and add it to the handler
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    
    # Add the handler to the logger
    logger.addHandler(file_handler)
    return logger

####  Setup & Logging ---

logger = setup_logging(log_file=os.path.join(OUTPUT_DIR, 'fl_train.log'))
logger.info(f"Starting FL Simulation (FedAvg) on {DEVICE}")
logger.info(f"Clients: {NUM_CLIENTS}, Rounds: {GLOBAL_ROUNDS}")
logger.info(f"Using Fixed Threshold from Baseline: {PREVIOUS_OPTIMAL_THRESHOLD}")

In [73]:
# ================= FIT TON-IoT NORMALIZATION (ADD THIS CELL) =================

import joblib

TONIOT_PATH = "/home/azwad/Works/IoMT_FL/Revised_Implementation/data/Ton_IoT/train_test_network.csv"   # <-- change if needed

df_full = pd.read_csv(TONIOT_PATH)

# --------- IMPORTANT ----------
# Train ONLY on benign traffic (unsupervised DAE requirement)
# ------------------------------
df_benign = df_full[df_full["label"] == 0].copy()

X_benign, encoder, feature_columns = preprocess_toniot_dataframe(
    df_benign,
    fit_encoder=True
)

scaler = StandardScaler()
scaler.fit(X_benign)

# Save so experiments are reproducible
joblib.dump(scaler, "toniot_scaler.joblib")
joblib.dump(encoder, "toniot_encoder.joblib")
joblib.dump(feature_columns, "toniot_columns.joblib")
print("TON-IoT scaler + encoder fitted successfully.")
print("Feature dimension:", X_benign.shape[1])

TON-IoT scaler + encoder fitted successfully.
Feature dimension: 35


In [74]:
# ================= LOAD TON-IoT DATASETS (REPLACE OLD DATASET CELL) =================

import torch
from torch.utils.data import Dataset

scaler = joblib.load("toniot_scaler.joblib")
encoder = joblib.load("toniot_encoder.joblib")
feature_columns = joblib.load("toniot_columns.joblib")

class TonIoTDataset(Dataset):
    def __init__(self, dataframe, scaler, encoder):

        self.labels = dataframe["label"].astype(int).values
        df = dataframe.drop(columns=["label"])

        X, _, _ = preprocess_toniot_dataframe(
                df,
                encoder=encoder,
                fit_encoder=False,
                feature_columns=feature_columns
            )

        self.features = scaler.transform(X)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# Split exactly same way you did for CIC
train_df = df_full[df_full["label"] == 0].copy()
test_df  = df_full.copy()

train_dataset = TonIoTDataset(train_df, scaler, encoder)
test_dataset  = TonIoTDataset(test_df, scaler, encoder)

print("Datasets ready:", len(train_dataset), len(test_dataset))

Datasets ready: 50000 211043


In [75]:
# ================= TON-IoT FEDERATED PARTITIONING =================

from torch.utils.data import DataLoader, Subset
import numpy as np

def get_client_loaders_from_dataset(train_dataset, test_dataset, num_clients, batch_size):

    # -------------------------------------------------
    # 1️⃣ Shuffle indices (federated simulation)
    # -------------------------------------------------
    num_samples = len(train_dataset)
    indices = np.random.permutation(num_samples)

    # Split indices equally across clients
    client_indices = np.array_split(indices, num_clients)

    client_loaders = []

    for idxs in client_indices:
        subset = Subset(train_dataset, idxs)

        loader = DataLoader(
            subset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )

        client_loaders.append(loader)

    # -------------------------------------------------
    # 2️⃣ Validation Loader (benign-only reconstruction check)
    # -------------------------------------------------
    val_loader = DataLoader(
        train_dataset,      # benign-only dataset
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # -------------------------------------------------
    # 3️⃣ Monitoring Loader (mixed traffic → F1/AUROC)
    # -------------------------------------------------
    monitor_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # -------------------------------------------------
    # 4️⃣ Input dimension (for DAE init)
    # -------------------------------------------------
    input_dim = train_dataset[0][0].shape[0]

    return client_loaders, val_loader, monitor_loader, input_dim

In [76]:
## Model Definition ###

class Encoder(nn.Module):

    def __init__(self, input_dim, latent_dim, dropout_p=0.1):
        super().__init__()
        
        # Main path
        self.main_path = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(32, latent_dim)
        )
        
        
        self.skip_path = nn.Linear(input_dim, latent_dim)

    def forward(self, x):
        
        return self.main_path(x) + self.skip_path(x)

class Decoder(nn.Module):

    def __init__(self, latent_dim, output_dim, dropout_p=0.1):
        super().__init__()
        
        # Main path
        self.main_path = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(64, output_dim)
        )
        

        self.skip_path = nn.Linear(latent_dim, output_dim)

    def forward(self, z):

        return self.main_path(z) + self.skip_path(z)

class DAE(nn.Module):

    def __init__(self, input_dim, latent_dim, noise_factor=0.1, dropout_p=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.noise_factor = noise_factor
        
        self.encoder = Encoder(
            input_dim=input_dim, 
            latent_dim=latent_dim, 
            dropout_p=dropout_p
        )
        
        self.decoder = Decoder(
            latent_dim=latent_dim, 
            output_dim=input_dim, 
            dropout_p=dropout_p
        )

    def forward(self, x):
        if self.training:
            noise = self.noise_factor * torch.randn_like(x)
            x_noisy = x + noise
        else:
            x_noisy = x
            

        z = self.encoder(x_noisy)
        

        x_recon = self.decoder(z)
        
        return x_recon



In [77]:
# --- Engine ---

class EarlyStopping:
    """Implements early stopping with patience."""
    def __init__(self, patience=10, verbose=False, delta=0, path='best_model.pth', logger=None):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.path = path
        self.logger = logger or logging.getLogger()
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.logger.info(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.logger.info(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def save_history(history, path='training_history.json'):
    """Saves the training history dictionary to a JSON file."""
    with open(path, 'w') as f:
        json.dump(history, f, indent=4)

def load_checkpoint(model, path='best_model.pth', device='cpu'):
    """Loads a model checkpoint from a file."""
    model.load_state_dict(torch.load(path, map_location=device))
    return model

def get_noise_factor(initial_noise, final_noise, epoch, total_epochs):
    """Anneals noise factor from initial to final (cosine schedule)."""
    
    cos_val = np.cos(np.pi * epoch / total_epochs)
    return final_noise + 0.5 * (initial_noise - final_noise) * (1 + cos_val)

def train_step(model, dataloader, criterion, optimizer, device, l1_lambda, current_noise_factor):
    """Performs a single training step (epoch)."""
    model.train()
    model.noise_factor = current_noise_factor # Set the noise factor
    
    total_recon_loss = 0.0
    total_l1_loss = 0.0
    
    for (features, _) in dataloader:
        features = features.to(device)
        
        optimizer.zero_grad()
        
        # --- DAE Forward Pass with L1 ---
        # 1. Add noise
        if model.training:
            noise = model.noise_factor * torch.randn_like(features)
            x_noisy = features + noise
        else:
            x_noisy = features # Should not happen in train_step, but safe
            
        # 2. Encode to get latent vector z
        z = model.encoder(x_noisy)
        
        # 3. Decode to get reconstruction
        x_recon = model.decoder(z)
        
        # --- Calculate Losses ---
        # 1. Reconstruction Loss (MSE)
        recon_loss = criterion(x_recon, features)
        
        # 2. L1 Sparsity Loss on latent vector
        l1_loss = l1_lambda * z.abs().mean()
        
        # 3. Total Loss
        loss = recon_loss + l1_loss
        
        loss.backward()
        optimizer.step()
        
        total_recon_loss += recon_loss.item()
        total_l1_loss += l1_loss.item()
        
    avg_recon_loss = total_recon_loss / len(dataloader)
    avg_l1_loss = total_l1_loss / len(dataloader)
    return avg_recon_loss, avg_l1_loss

def val_step(model, dataloader, criterion, device):
    """Performs a single validation step (epoch)."""
    model.eval()
    total_recon_loss = 0.0
    
    with torch.no_grad():
        for (features, _) in dataloader:
            features = features.to(device)
            
            # Forward pass (no noise, no L1)
            x_recon = model(features)
            
            # Reconstruction Loss
            recon_loss = criterion(x_recon, features)
            total_recon_loss += recon_loss.item()
            
    avg_recon_loss = total_recon_loss / len(dataloader)
    return avg_recon_loss

# --- Thresholding and Testing Logic ---

def find_threshold(model, dataloader, device, percentile=95):
    """
    Finds the reconstruction error threshold from the validation set.
    """
    model.eval()
    # Use MSELoss with 'none' to get per-sample errors
    criterion = nn.MSELoss(reduction='none')
    all_errors = []
    
    with torch.no_grad():
        for (features, _) in dataloader:
            features = features.to(device)
            x_recon = model(features)
            
            # Calculate per-sample error
            errors = criterion(x_recon, features).mean(dim=1)
            all_errors.append(errors.cpu().numpy())
            
    all_errors = np.concatenate(all_errors)
    
    # Set threshold at the nth percentile
    threshold = np.percentile(all_errors, percentile)
    return threshold

def test_model(model, dataloader, device, threshold, benign_label, label_encoder_classes):
    """
    Tests the model on the balanced test set and computes all metrics.
    """
    model.eval()
    criterion = nn.MSELoss(reduction='none')
    
    all_y_true = []
    all_y_true_binary = []
    all_y_pred_binary = []
    all_y_scores = []

    with torch.no_grad():
        for (features, labels) in dataloader:
            features = features.to(device)
            
            x_recon = model(features)
            
            # Per-sample reconstruction error (our anomaly score)
            errors = criterion(x_recon, features).mean(dim=1).cpu().numpy()
            
            # Predicted labels (1 = Attack, 0 = Benign)
            preds = (errors > threshold).astype(int)
            
            # True labels (binary)
            labels_np = labels.cpu().numpy()
            true_binary = (labels_np != benign_label).astype(int)
            
            all_y_true.extend(labels_np)
            all_y_true_binary.extend(true_binary)
            all_y_pred_binary.extend(preds)
            all_y_scores.extend(errors)

    # --- Calculate All Metrics ---
    results = {}
    
    y_true_b = np.array(all_y_true_binary)
    y_pred_b = np.array(all_y_pred_binary)
    y_scores = np.array(all_y_scores)
    y_true_multi = np.array(all_y_true)
    
    results['AUROC'] = roc_auc_score(y_true_b, y_scores)
    results['AUPRC'] = average_precision_score(y_true_b, y_scores)
    results['F1_Score'] = f1_score(y_true_b, y_pred_b)
    results['Precision'] = precision_score(y_true_b, y_pred_b)
    results['Recall'] = recall_score(y_true_b, y_pred_b)
    results['Accuracy'] = accuracy_score(y_true_b, y_pred_b)
    results['MCC'] = matthews_corrcoef(y_true_b, y_pred_b)
    results['Balanced_Accuracy'] = balanced_accuracy_score(y_true_b, y_pred_b)
    
    # TPR at 1% FPR
    fpr, tpr, _ = roc_curve(y_true_b, y_scores)
    results['TPR_at_1_FPR'] = np.interp(0.01, fpr, tpr)
    
    # FPR at 95% TPR
    # We need to sort by TPR to use interp
    sort_idx = np.argsort(tpr)
    tpr_sorted = tpr[sort_idx]
    fpr_sorted = fpr[sort_idx]
    results['FPR_at_95_TPR'] = np.interp(0.95, tpr_sorted, fpr_sorted)
    
    # Per-attack class recall
    per_class_recall = {}
    attack_labels = [i for i, cls in enumerate(label_encoder_classes) if cls not in ['Benign', 'benign']]
    
    for label_int in attack_labels:
        class_name = label_encoder_classes[label_int]
        
        # Get mask for samples of this specific attack class
        class_mask = (y_true_multi == label_int)
        
        if np.sum(class_mask) == 0:
            per_class_recall[class_name] = "N/A (No samples in test set)"
            continue
            
        class_y_true = y_true_b[class_mask]
        class_y_pred = y_pred_b[class_mask]
        
        # Calculate recall for this class (all should be '1')
        class_recall = recall_score(class_y_true, class_y_pred, zero_division=0)
        per_class_recall[class_name] = class_recall
        
    results['Per_Attack_Recall'] = per_class_recall

    return results

In [78]:
# Initialize Global Model ---
global_model = DAE(
    input_dim=INPUT_DIM, 
    latent_dim=LATENT_DIM, 
    noise_factor=INITIAL_NOISE, 
    dropout_p=DROPOUT_P
).to(DEVICE)


In [79]:
# Helper Functions ---

def fed_avg(weights_list, sample_counts):
    total_samples = sum(sample_counts)
    global_weights = copy.deepcopy(weights_list[0])
    for key in global_weights.keys():
        global_weights[key] = torch.zeros_like(global_weights[key], dtype=torch.float32)
        
    for weights, count in zip(weights_list, sample_counts):
        weight_factor = count / total_samples
        for key in weights.keys():
            global_weights[key] += weights[key] * weight_factor
    return global_weights

def calculate_communication_cost(model):
    """Estimates model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_all_mb = (param_size + buffer_size) / 1024**2
    return size_all_mb

# Calculate Communication overhead
model_size_mb = calculate_communication_cost(global_model)
logger.info(f"Global Model Size: {model_size_mb:.4f} MB")
# Cost per round = (Broadcast + Upload) * Num_Clients
comm_cost_per_round = model_size_mb * 2 * NUM_CLIENTS 

best_model_path = os.path.join(OUTPUT_DIR, 'global_best_model.pth')
early_stopper = EarlyStopping(
    patience=PATIENCE, verbose=True, path=best_model_path, logger=logger
)
criterion = nn.MSELoss()

# History tracker with FL metrics
history = {
    'round': [], 
    'train_loss': [], 
    'val_loss': [], 
    'val_f1': [], 
    'val_auroc': [],
    'cumulative_comm_mb': []
}

def evaluate_global_metrics(model, dataloader, threshold, benign_label):
    """Quick evaluation for F1 and AUROC during training rounds."""
    model.eval()
    criterion = nn.MSELoss(reduction='none')
    all_preds = []
    all_labels = []
    all_scores = []
    
    with torch.no_grad():
        for features, labels in dataloader:
            features = features.to(DEVICE)
            x_recon = model(features)
            errors = criterion(x_recon, features).mean(dim=1).cpu().numpy()
            
            preds = (errors > threshold).astype(int)
            # Convert labels to binary (0=Benign, 1=Attack)
            binary_labels = (labels.cpu().numpy() != benign_label).astype(int)
            
            all_preds.extend(preds)
            all_labels.extend(binary_labels)
            all_scores.extend(errors)
            
    f1 = f1_score(all_labels, all_preds)
    auroc = roc_auc_score(all_labels, all_scores)
    return f1, auroc

In [80]:
# --- 7. FL Training Loop ---
logger.info("\n--- Starting Federated Training ---")
start_time = time.time()
client_loaders, global_val_loader, global_monitor_loader, input_dim = \
    get_client_loaders_from_dataset(
        train_dataset,
        test_dataset,
        NUM_CLIENTS,
        BATCH_SIZE
    )

bulyan_log = []
for round_idx in range(GLOBAL_ROUNDS):
    round_start = time.time()
    current_noise = get_noise_factor(INITIAL_NOISE, FINAL_NOISE, round_idx, GLOBAL_ROUNDS)
    
    local_weights = []
    local_losses = []
    client_sample_counts = []
    
    # --- Client Phase ---
    for client_idx, loader in enumerate(client_loaders):
        local_model = copy.deepcopy(global_model)
        local_model.train()
        local_model.noise_factor = current_noise
        
        optimizer = AdamW(local_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        
        client_loss_accum = 0.0
        for epoch in range(CLIENT_EPOCHS_PER_ROUND):
            r_loss, l1_loss = train_step(
                local_model, loader, criterion, optimizer, DEVICE, L1_LAMBDA, current_noise
            )
            client_loss_accum += (r_loss + l1_loss)
        
        avg_client_loss = client_loss_accum / CLIENT_EPOCHS_PER_ROUND
        local_losses.append(avg_client_loss)
        local_weights.append(copy.deepcopy(local_model.state_dict()))
        client_sample_counts.append(len(loader.dataset))
    
    # --- Server Phase ---
    # Aggregation
    avg_train_loss = sum(local_losses) / len(local_losses)
    new_global_weights = fed_avg(local_weights,client_sample_counts)
    global_model.load_state_dict(new_global_weights)
    
    # Validation (Recon Loss on Benign)
    val_recon_loss = val_step(global_model, global_val_loader, criterion, DEVICE)
    
    # Metrics (F1/AUROC on Mixed Set using Fixed Threshold)
    val_f1, val_auroc = evaluate_global_metrics(
        global_model, global_monitor_loader, PREVIOUS_OPTIMAL_THRESHOLD, 0
    )
    
    # Track History
    total_comm = comm_cost_per_round * (round_idx + 1)
    history['round'].append(round_idx + 1)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_recon_loss)
    history['val_f1'].append(val_f1)
    history['val_auroc'].append(val_auroc)
    history['cumulative_comm_mb'].append(total_comm)
    
    logger.info(f"Round {round_idx+1}/{GLOBAL_ROUNDS} | "
                f"Loss(Val): {val_recon_loss:.6f} | "
                f"F1: {val_f1:.4f} | AUROC: {val_auroc:.4f} | "
                f"Comm: {total_comm:.2f} MB")
    
    bulyan_log.append({
    "round": round_idx + 1,
    "val_loss": float(val_recon_loss),
    "f1": float(val_f1)
        })
            
    # Checkpointing (Based on Benign Reconstruction Loss - standard for DAE)
    early_stopper(val_recon_loss, global_model)
    if early_stopper.early_stop:
        logger.info(f"Early stopping triggered at round {round_idx+1}")
        break

total_training_time = time.time() - start_time
logger.info(f"--- FL Training Complete in {total_training_time:.2f} seconds ---")

# Save History
history['total_training_time_sec'] = total_training_time
save_history(history, os.path.join(OUTPUT_DIR, 'fl_history.json'))

In [82]:
# --- 8. Final Evaluation ---
logger.info("\n--- Starting Final Evaluation (Test Set) ---")

# Load best global model
global_model = load_checkpoint(global_model, best_model_path, DEVICE)
CLASS_NAMES = ["Benign", "Attack"]
# Reuse the global_monitor_loader (which is test_balanced) for final detailed metrics
results = test_model(
    global_model, 
    global_monitor_loader, 
    DEVICE, 
    PREVIOUS_OPTIMAL_THRESHOLD, # Use fixed threshold
    0, 
    CLASS_NAMES
)

# Add FL Specific Metrics to Results
results['Total_Communication_MB'] = history['cumulative_comm_mb'][-1]
results['Total_Training_Time_Sec'] = total_training_time
results['Rounds_Completed'] = len(history['round'])

# Log Results
logger.info("\n--- FL GLOBAL TEST RESULTS ---")
main_metrics = {k: v for k, v in results.items() if k != 'Per_Attack_Recall'}
for metric, value in main_metrics.items():
    logger.info(f"{metric:<25}: {value}")

logger.info("\n--- PER-ATTACK-CLASS RECALL ---")
per_class_recall = results['Per_Attack_Recall']
for class_name, recall in per_class_recall.items():
    logger.info(f"{class_name:<25}: {recall}")

# Save Results JSON
with open(os.path.join(OUTPUT_DIR, 'fl_test_results.json'), 'w') as f:
    json.dump(results, f, indent=4)

logger.info(f"Results saved to {OUTPUT_DIR}")

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x75ca3914bd80><function _MultiProcessingDataLoaderIter.__del__ at 0x75ca3914bd80>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/azwad/anaconda3/envs/IoMT_FL/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
  File "/home/azwad/anaconda3/envs/IoMT_FL/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/home/azwad/anaconda3/envs/IoMT_FL/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
  File "/home/azwad/anaconda3/envs/IoMT_FL/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
        if w.is_alive():if w.is_alive():

              ^^^^^^^^^^^^^^^^^^^^^^^^

  File "/home/azwad/anaconda3/envs/IoMT_FL/lib/python3.11/multiprocessing/process.